# Multimodal AI

## What is Multimodal AI?
Multimodal AI processes and generates content across multiple modalities: text, images, audio, video, 3D, tabular data.

## Fusion Strategies

### Early Fusion
Combine raw inputs before encoding:
$$z = f([x_{text}; x_{image}; x_{audio}])$$

### Late Fusion
Encode each modality separately, then combine:
$$z = g(f_1(x_{text}), f_2(x_{image}), f_3(x_{audio}))$$

### Cross-Attention Fusion (Flamingo)
Text tokens attend to visual tokens via cross-attention:
$$\text{CrossAttn}(Q_{text}, K_{image}, V_{image}) = \text{softmax}\left(\frac{Q_{text}K_{image}^T}{\sqrt{d_k}}\right)V_{image}$$

## CLIP Contrastive Loss

CLIP trains image and text encoders jointly:

$$L = -\frac{1}{N}\sum_i \log \frac{e^{s(I_i, T_i)/\tau}}{\sum_j e^{s(I_i, T_j)/\tau}}$$

where $s(I, T)$ is cosine similarity, $\tau$ is temperature.

## Modality Overview

| Modality | Input | Output Models |
|----------|-------|---------------|
| Text | Tokens | GPT, Claude, Llama |
| Image | Patches/pixels | ViT, CLIP, DinoV2 |
| Audio | Spectrograms | Whisper, wav2vec |
| Video | Frame sequences | VideoLLaMA, Sora |
| 3D | Point clouds | PointNet, 3D-LLM |
| Tabular | Encoded features | TabPFN, TabTransformer |

In [1]:
# CLIP Image and Text in shared embedding space
# pip install transformers torch Pillow
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import requests
from io import BytesIO

model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Zero-shot image classification
def clip_classify(image, candidate_labels):
    inputs = processor(
        text=candidate_labels,
        images=image,
        return_tensors="pt",
        padding=True
    )
    with torch.no_grad():
        outputs = model(**inputs)

    probs = outputs.logits_per_image.softmax(dim=1)[0]
    return dict(zip(candidate_labels, probs.tolist()))

# Create a simple test image
from PIL import Image as PILImage
import numpy as np
img_array = np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8)
test_image = PILImage.fromarray(img_array)

labels = ["a photo of a cat", "a photo of a dog", "a landscape photo", "a photo of food"]
result = clip_classify(test_image, labels)
print('CLIP zero-shot classification:')
for label, prob in sorted(result.items(), key=lambda x: -x[1]):
    print(f'  {prob:.3f} {label}')

CLIP zero-shot classification:
  0.446 a photo of a dog
  0.399 a photo of a cat
  0.140 a photo of food
  0.014 a landscape photo


In [2]:
# ImageBind 6 modalities in one embedding space
IMAGEBIND_CODE = '''
# pip install imagebind
from imagebind import data
from imagebind.models import imagebind_model
from imagebind.models.imagebind_model import ModalityType
import torch

model = imagebind_model.imagebind_huge(pretrained=True)
model.eval()

inputs = {
    ModalityType.TEXT:  data.load_and_transform_text(["A dog playing fetch"], device="cpu"),
    ModalityType.VISION: data.load_and_transform_vision_data(["dog.jpg"], device="cpu"),
    ModalityType.AUDIO: data.load_and_transform_audio_data(["dog_bark.wav"], device="cpu"),
}

with torch.no_grad():
    embeddings = model(inputs)

# All modalities in the same 1024-dim space
# Can compute similarity across ANY modalities:
text_emb = embeddings[ModalityType.TEXT]     # (1, 1024)
vision_emb = embeddings[ModalityType.VISION]  # (1, 1024)
audio_emb = embeddings[ModalityType.AUDIO]    # (1, 1024)

sim_tv = torch.cosine_similarity(text_emb, vision_emb)
sim_ta = torch.cosine_similarity(text_emb, audio_emb)
print(f"Text-Vision similarity: {sim_tv.item():.3f}")
print(f"Text-Audio similarity:  {sim_ta.item():.3f}")
'''
print(IMAGEBIND_CODE)


# pip install imagebind
from imagebind import data
from imagebind.models import imagebind_model
from imagebind.models.imagebind_model import ModalityType
import torch

model = imagebind_model.imagebind_huge(pretrained=True)
model.eval()

inputs = {
    ModalityType.TEXT:  data.load_and_transform_text(["A dog playing fetch"], device="cpu"),
    ModalityType.VISION: data.load_and_transform_vision_data(["dog.jpg"], device="cpu"),
    ModalityType.AUDIO: data.load_and_transform_audio_data(["dog_bark.wav"], device="cpu"),
}

with torch.no_grad():
    embeddings = model(inputs)

# All modalities in the same 1024-dim space
# Can compute similarity across ANY modalities:
text_emb = embeddings[ModalityType.TEXT]     # (1, 1024)
vision_emb = embeddings[ModalityType.VISION]  # (1, 1024)
audio_emb = embeddings[ModalityType.AUDIO]    # (1, 1024)

sim_tv = torch.cosine_similarity(text_emb, vision_emb)
sim_ta = torch.cosine_similarity(text_emb, audio_emb)
print(f"Text-Vision similarity: {sim_tv.item

In [3]:
# Multimodal RAG combining text, images, tables
MULTIMODAL_RAG_CODE = '''
# pip install unstructured[pdf] langchain-openai
from unstructured.partition.pdf import partition_pdf
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
import base64

# Extract text, tables, images from PDF
elements = partition_pdf(
    filename="document.pdf",
    extract_images_in_pdf=True,
    infer_table_structure=True,
    chunking_strategy="by_title",
    max_characters=4000,
)

# Summarize images using GPT-4o
def summarize_image(image_path):
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    client = ChatOpenAI(model="gpt-4o")
    return client.invoke([{"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
                          {"type": "text", "text": "Describe this image for indexing:"}]).content

# Store text + image summaries in vector store
texts = [e.text for e in elements if hasattr(e, 'text')]
vectorstore = Chroma.from_texts(texts, OpenAIEmbeddings())
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
'''
print(MULTIMODAL_RAG_CODE)


# pip install unstructured[pdf] langchain-openai
from unstructured.partition.pdf import partition_pdf
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_chroma import Chroma
import base64

# Extract text, tables, images from PDF
elements = partition_pdf(
    filename="document.pdf",
    extract_images_in_pdf=True,
    infer_table_structure=True,
    chunking_strategy="by_title",
    max_characters=4000,
)

# Summarize images using GPT-4o
def summarize_image(image_path):
    with open(image_path, "rb") as f:
        b64 = base64.b64encode(f.read()).decode()
    client = ChatOpenAI(model="gpt-4o")
    return client.invoke([{"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{b64}"}},
                          {"type": "text", "text": "Describe this image for indexing:"}]).content

# Store text + image summaries in vector store
texts = [e.text for e in elements if hasattr(e, 'text')]
vectorstore = Chroma.from_texts(texts, OpenAIEmbeddings())
retrieve

In [4]:
# Video understanding with Gemini
VIDEO_CODE = '''
import google.generativeai as genai
import time

genai.configure(api_key="GEMINI_API_KEY")
model = genai.GenerativeModel("gemini-1.5-pro")

# Upload video file
video_file = genai.upload_file("video.mp4")
while video_file.state.name == "PROCESSING":
    time.sleep(2)
    video_file = genai.get_file(video_file.name)

# Ask questions about the video
response = model.generate_content([
    video_file,
    "Summarize the key events in this video."
])
print(response.text)

# Gemini supports up to 1 hour of video / 1M tokens context
'''
print(VIDEO_CODE)


import google.generativeai as genai
import time

genai.configure(api_key="GEMINI_API_KEY")
model = genai.GenerativeModel("gemini-1.5-pro")

# Upload video file
video_file = genai.upload_file("video.mp4")
while video_file.state.name == "PROCESSING":
    time.sleep(2)
    video_file = genai.get_file(video_file.name)

# Ask questions about the video
response = model.generate_content([
    video_file,
    "Summarize the key events in this video."
])
print(response.text)

# Gemini supports up to 1 hour of video / 1M tokens context



## Additional Learning Resources

### Papers
- [CLIP](https://arxiv.org/abs/2103.00020) Radford et al., 2021
- [Flamingo](https://arxiv.org/abs/2204.14198) Alayrac et al., 2022
- [ImageBind](https://arxiv.org/abs/2305.05665) Girdhar et al., 2023
- [GPT-4V Technical Report](https://arxiv.org/abs/2303.08774)
- [Gemini Technical Report](https://arxiv.org/abs/2312.11805)
- [VideoLLaMA](https://arxiv.org/abs/2306.02858)
- [MusicGen](https://arxiv.org/abs/2306.05284)

### Tools
- [Hugging Face Multimodal Models](https://huggingface.co/models?pipeline_tag=image-text-to-text)
- [unstructured.io](https://unstructured.io/) PDF/doc parsing
- [LangChain Multimodal](https://python.langchain.com/docs/how_to/multimodal_inputs/)
- [ImageBind GitHub](https://github.com/facebookresearch/ImageBind)
- [diffusers](https://github.com/huggingface/diffusers) image/video generation